In [1]:
pip install transformers torch torchaudio datasets sentencepiece

In [2]:
import torch
import torchaudio
from transformers import (
    AutoProcessor,
    AutoModelForCTC,
    pipeline
)

In [3]:
# Load ASR Model
asr_model_id = "janiduchamika/wav2vec2-xls-r-300m-sinhala-general-185k"

asr_processor = AutoProcessor.from_pretrained(asr_model_id)
asr_model     = AutoModelForCTC.from_pretrained(asr_model_id)
asr_model.eval()

preprocessor_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/30.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

Wav2Vec2ForCTC(
  (wav2vec2): Wav2Vec2Model(
    (feature_extractor): Wav2Vec2FeatureEncoder(
      (conv_layers): ModuleList(
        (0): Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(1, 512, kernel_size=(10,), stride=(5,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (1-4): 4 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(3,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
        (5-6): 2 x Wav2Vec2LayerNormConvLayer(
          (conv): Conv1d(512, 512, kernel_size=(2,), stride=(2,))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
          (activation): GELUActivation()
        )
      )
    )
    (feature_projection): Wav2Vec2FeatureProjection(
      (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
      (projec

In [4]:
# Load Sentiment Model-->twitter-xlm-roberta-base
sentiment_pipe = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-xlm-roberta-base-sentiment
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [5]:
# Load and Resample Audio
def load_audio(path, target_sr=16000):
    waveform, sr = torchaudio.load(path)
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform  = resampler(waveform)
    return waveform.squeeze().numpy()

In [6]:
# HELPER: ASR- Audio-->Raw text
def transcribe(audio_path):
    audio = load_audio(audio_path)
    inputs = asr_processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    return asr_processor.batch_decode(predicted_ids)[0]

In [7]:
# HELPER: Sentiment Analysis
def analyse_sentiment(text):
    # model max token limit is 512, truncate if needed
    result = sentiment_pipe(text[:512])
    return result[0]

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Test pipeline
audio_file = "/content/drive/MyDrive/Sentiment Analysis/test.wav"  # ← update path if needed

raw_transcript = transcribe(audio_file)
sentiment      = analyse_sentiment(raw_transcript)

print("Raw ASR output :", raw_transcript)
print("Sentiment Label:", sentiment["label"])   # Positive / Negative / Neutral
print("Confidence     :", round(sentiment["score"], 4))

Raw ASR output : අයිබවන් ම ප්‍රබත්මට පුළුවනි ොබර සහය වන්න ේංලින්දතකරබල ්බර්මදවස්පුනරේිවසෙක්් දසිතුන්නිමමලයල්දෙසැ්බර් විස් සිතුම්වෙනිද ඉඳලා නද විසිදරේහුකනක්සඑක හද දුන්නහැදන මවදසිබගේකදුවට පං ඔලිවව මටකක්ඔහටල්ක මේාධාර විදිහට අදෙවන්මට පුල්කමදාහරි ගැටළුව වෙලා තියෙන් ලෙස මේ ඇතුළත් කරපුවන්ක තුළමද නම කොහොමද  තේර්න මේකනක්ෂණිකමරයම විස්තර පරීක්ෂා කිරීමක් කරලා බැලුවස පහුගිය දෙසැ්බර් මාස විසි හත්වෙනිදා දිනේ තුළ ම ගැටුව පළිඳව පර කම්ප්ලේන එකක් දලා තියෙනවා  සාමාන්‍යයෙන් බිල්පත් අඩු කිරීමක ගැටළුව තිබුණු මාසේට අදාල බ්පතහරහාම වෙන්නේ නැැ එහෙ කරන්න හේතුව තියෙන්න ඒ විදිහටඅප බිල්පතරහා මුදලඅඩුකරොත්ගේ බදු මුදල අඩුවීම න්නේ  සාමා්‍ය ිල්පත් නිකුත් කරීමේ ක්‍රමය වේනේ දැන් ගැටළුව අවසානුනේස දැම්බර් මාසය තු දී  තකොට ඇයි ඊගාවර නිකුත් වෙන ිල්පත හරහා තමයි සැගේ දෙසැම්බර් මසේ වගේම නොවැම්බර් මාසේකණික්ෂ එක වැඩ නොකරපු කාල සීමා වේ බිල්පත් අඩු කිරීම වෙන්නඑතොට සකශනවාරි බිල්පතසගබලන් සයි කර් ෙකට අනුව තෙබරවාරි පළවිනිදා හෝ දෙවනිදා කියන ද දෙක තුළ නිකුත් වෙනා ඒබිල්පයන් තමයි ජස්මන් එක ලැබෙන් නෙස අ්මන්ට ක ලබාදීම සම්බන්ධව කිසිම ගැටළුවක් 